In [ ]:

# ============================================================
# ALBUM ART LOOKUP PIPELINE — TEST VERSION
# ============================================================
#
# Purpose:
#   1. Read the first three albums from PostgreSQL.
#   2. Search MusicBrainz for matching releases.
#   3. Rank candidates using fuzzy title/artist matching.
#   4. Accept the highest-scoring candidate if score >= 75.
#   5. Check whether its front-cover artwork is available.
#   6. Display the results without modifying PostgreSQL.
#
# Database table: merged23_25
# Internal key:   id
# Album title:    album_name
# Artist:         artist_name
#
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from rapidfuzz import fuzz
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

# Set your database connection string in the DATABASE_URL
# environment variable before running this notebook.

load_dotenv()

database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise RuntimeError("DATABASE_URL is not configured.")

engine = create_engine(database_url)

df = pd.read_sql("SELECT * FROM merged23_25", engine)

print(df.head(5))

# MusicBrainz release-search API.
MB_URL = "https://musicbrainz.org/ws/2/release/"

# Cover Art Archive API.
CAA_URL = "https://coverartarchive.org/release/"

# Minimum combined fuzzy-matching score accepted.
MATCH_THRESHOLD = 75

# Number of albums to test in this run.
TEST_LIMIT = 3

# Replace with a real contact email for your application.
USER_AGENT_EMAIL = os.getenv("MUSICBRAINZ_USER_AGENT_EMAIL")

if not USER_AGENT_EMAIL:
    raise RuntimeError("MUSICBRAINZ_USER_AGENT_EMAIL is not configured in .env")

USER_AGENT = f"AlbumArtResearch/1.0 ({USER_AGENT_EMAIL})"

# Create a reusable HTTP session.
session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT
})


# ------------------------------------------------------------
# 3. MUSICBRAINZ REQUEST HELPER
# ------------------------------------------------------------


def musicbrainz_get(url, params=None, max_retries=4):
    """
    Send a MusicBrainz request with rate limiting and retries.

    Retries HTTP 429, 500, 502, 503, and 504 responses.
    Honors Retry-After when it contains a valid positive delay.
    Raises an exception if all attempts fail.
    """

    retryable_codes = (429, 500, 502, 503, 504)

    for attempt in range(max_retries):

        # Respect MusicBrainz rate limits.
        time.sleep(1.2)

        try:
            response = session.get(
                url,
                params=params,
                timeout=30
            )

            if response.status_code in retryable_codes:

                # Stop if this was the final attempt.
                if attempt == max_retries - 1:
                    response.raise_for_status()

                retry_after = response.headers.get("Retry-After")

                try:
                    wait_seconds = float(retry_after)

                    if wait_seconds <= 0:
                        raise ValueError(
                            "Retry-After must be positive"
                        )

                except (ValueError, TypeError):
                    wait_seconds = 10 * (2 ** attempt)

                print(
                    f"MusicBrainz HTTP "
                    f"{response.status_code}. "
                    f"Retrying after "
                    f"{wait_seconds:.0f} seconds."
                )

                time.sleep(wait_seconds)
                continue

            # Raise for other unsuccessful responses.
            response.raise_for_status()

            return response

        except requests.RequestException as exc:

            if attempt == max_retries - 1:
                raise

            wait_seconds = 10 * (2 ** attempt)

            print(
                f"MusicBrainz request failed: {exc}. "
                f"Retrying in {wait_seconds:.0f} seconds."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "MusicBrainz request failed after all retries."
    )


def find_album_art(album_name, artist_name):
    """
    Search MusicBrainz for releases, exclude releases without
    front-cover artwork, then rank the remaining candidates.

    Does not modify PostgreSQL.
    """

    # 1. Search MusicBrainz.
    query = (
        f'release:"{album_name}" AND '
        f'artist:"{artist_name}"'
    )

    response = musicbrainz_get(
        MB_URL,
        params={
            "query": query,
            "fmt": "json",
            "limit": 25
        }
    )

    releases = response.json().get("releases", [])

    if not releases:
        return {"status": "not_found"}

    # 2. Filter out releases without available cover art
    # BEFORE calculating any fuzzy-matching scores.
    releases_with_art = []

    for release in releases:
        release_id = release.get("id")

        if not release_id:
            continue

        artwork_url = f"{CAA_URL}{release_id}/front-500"

        try:
            cover_response = session.get(
                artwork_url,
                timeout=30,
                stream=True,
                allow_redirects=True
            )

            try:
                if cover_response.status_code == 404:
                    continue

                cover_response.raise_for_status()

            finally:
                cover_response.close()

        except requests.RequestException as exc:
            # Do not mistake a service failure for missing art.
            return {
                "status": "artwork_check_error",
                "release_id": release_id,
                "error": str(exc)
            }

        releases_with_art.append({
            "release": release,
            "artwork_url": artwork_url
        })

        # Conservative pacing for artwork requests.
        time.sleep(1.0)

    if not releases_with_art:
        return {"status": "no_artwork"}

    # 3. Calculate fuzzy scores ONLY for releases with artwork.
    candidates = []

    for item in releases_with_art:
        release = item["release"]

        candidate_title = release.get("title", "")
        candidate_artist = get_credited_artist(release)

        title_score = fuzz.ratio(
            str(album_name).casefold().strip(),
            candidate_title.casefold().strip()
        )

        artist_score = fuzz.ratio(
            str(artist_name).casefold().strip(),
            candidate_artist.casefold().strip()
        )

        combined_score = (
            0.6 * title_score + 0.4 * artist_score
        )

        candidates.append({
            "release": release,
            "artwork_url": item["artwork_url"],
            "title_score": title_score,
            "artist_score": artist_score,
            "score": combined_score
        })

    # 4. Rank only candidates that have artwork.
    candidates.sort(
        key=lambda candidate: candidate["score"],
        reverse=True
    )

    best = candidates[0]

    # 5. Apply the matching threshold.
    if best["score"] < MATCH_THRESHOLD:
        return {
            "status": "below_threshold",
            "release_id": best["release"].get("id"),
            "matched_title": best["release"].get("title"),
            "matched_artist": get_credited_artist(
                best["release"]
            ),
            "title_score": round(best["title_score"], 2),
            "artist_score": round(best["artist_score"], 2),
            "score": round(best["score"], 2)
        }

    # 6. Return the highest-scoring candidate with artwork.
    return {
        "status": "matched",
        "release_id": best["release"].get("id"),
        "matched_title": best["release"].get("title"),
        "matched_artist": get_credited_artist(
            best["release"]
        ),
        "artwork_url": best["artwork_url"],
        "title_score": round(best["title_score"], 2),
        "artist_score": round(best["artist_score"], 2),
        "score": round(best["score"], 2)
    }

# ------------------------------------------------------------
# 4. ARTIST-CREDIT EXTRACTION
# ------------------------------------------------------------

def get_credited_artist(release):
    """
    Extract the credited artist string from a MusicBrainz
    release result.

    MusicBrainz artist credits may contain multiple artists
    and joining text such as ' & ' or ' feat. '.
    """

    artist_parts = []

    for item in release.get("artist-credit", []):

        if isinstance(item, dict):

            # Prefer the credited name, falling back to the
            # nested artist name if needed.
            artist_name = item.get(
                "name",
                item.get("artist", {}).get("name", "")
            )

            artist_parts.append(artist_name)
            artist_parts.append(item.get("joinphrase", ""))

        elif isinstance(item, str):
            artist_parts.append(item)

    return "".join(artist_parts).strip()


# ------------------------------------------------------------
# 5. FIND ALBUM ART
# ------------------------------------------------------------


# ------------------------------------------------------------
# 6. READ THE FIRST THREE ALBUMS FROM POSTGRESQL
# ------------------------------------------------------------

# We select only the fields needed for the lookup.
# LIMIT 3 makes this a controlled test.

with engine.connect() as conn:

    test_albums = pd.read_sql(
        text("""
            SELECT id, album_name, artist_name
            FROM merged23_25
            ORDER BY id
            LIMIT :limit
        """),
        conn,
        params={"limit": TEST_LIMIT}
    )

print(f"Loaded {len(test_albums)} albums for testing.")

display(test_albums)




   id         album_name     artist_name  score  \
0   1              Lahai          Sampha    8.5   
1   2  melodic nostalgia       tomcbumpz    5.0   
2   3       OK ORCHESTRA             AJR    1.0   
3   4           HELLMODE  Jeff Rosentock    8.0   
4   5     Dorothy's Harp   Dorothy Ashby    9.5   

                                               notes  listen_year artwork_url  \
0  A smooth R&B infused hip hop album. Opener is ...         2023        None   
1  This is a lowfi beats to study to ep. I didn't...         2023        None   
2  I decided I need to suffer I guess. AJR is the...         2023        None   
3  The dying masterpiece of California garage pun...         2023        None   
4  A funky bass and drum beat lay the ground for ...         2023        None   

  musicbrainz_release_id artwork_status  
0                   None        pending  
1                   None        pending  
2                   None        pending  
3                   None        pendin

,id,album_name,artist_name
0,1,Lahai,Sampha
1,2,melodic nostalgia,tomcbumpz
2,3,OK ORCHESTRA,AJR


In [2]:
# ------------------------------------------------------------
# 7. LOOK UP ARTWORK FOR EACH ALBUM
# ------------------------------------------------------------

results = []

for row in test_albums.itertuples(index=False):

    print(
        f"\nSearching: {row.artist_name} — {row.album_name}"
    )

    try:

        # Perform the search and artwork check.
        match = find_album_art(
            row.album_name,
            row.artist_name
        )

        # Preserve the internal PostgreSQL ID so each result
        # can later be mapped back to the correct album row.
        results.append({
            "id": row.id,
            "album_name": row.album_name,
            "artist_name": row.artist_name,
            **match
        })

        print("Status:", match["status"])

        if match.get("score") is not None:
            print("Combined score:", match["score"])

    except requests.RequestException as exc:

        # Preserve failed requests as errors.
        # A network failure is not the same as missing artwork.
        results.append({
            "id": row.id,
            "album_name": row.album_name,
            "artist_name": row.artist_name,
            "status": "error",
            "error": str(exc)
        })

        print("Request error:", exc)


# ------------------------------------------------------------
# 8. DISPLAY THE RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\nArtwork lookup results:")

display(results_df)



# ------------------------------------------------------------
# 9. REVIEW THE THREE-ALBUM TEST
# ------------------------------------------------------------

print("Lookup status summary:")
display(
    results_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)

# Show the columns most useful for checking matches.
review_columns = [
    "id",
    "album_name",
    "artist_name",
    "status",
    "matched_title",
    "matched_artist",
    "release_id",
    "artwork_url",
    "score"
]

# Only select columns that exist in this run's results.
review_columns = [
    column for column in review_columns
    if column in results_df.columns
]

display(results_df[review_columns])

print("\nTest complete. PostgreSQL was not modified.")




Searching: Sampha — Lahai


KeyboardInterrupt: 

In [ ]:
for _, row in results_df[
    results_df["status"] == "matched"
].iterrows():

    print(
        f'{row["artist_name"]} — {row["album_name"]}: '
        f'{row["artwork_url"]}'
    )

Sampha — Lahai: https://coverartarchive.org/release/2530be1a-de66-4e1c-89c8-b25d4552c2b0/front-500
AJR — OK ORCHESTRA: https://coverartarchive.org/release/8b7f3e12-50d5-4441-86a9-84ae02fa507d/front-500


In [ ]:
# Prepare album list
albums_to_process = df[
    ["id", "album_name", "artist_name"]
].copy()

OUTPUT_DIR = "album_art_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CHUNKS = 10
MAX_WORKERS = 5


def process_one_album(row):
    """Process one album and return its result."""

    result = {
        "id": row.id,
        "album_name": row.album_name,
        "artist_name": row.artist_name,
    }

    if pd.isna(row.album_name) or pd.isna(row.artist_name):
        result["status"] = "missing_input"
        return result

    try:
        album_data = find_album_art(
            row.album_name,
            row.artist_name
        )
        result.update(album_data)

    except Exception as exc:
        result.update({
            "status": "error",
            "error": str(exc)
        })

    return result


def process_album_chunk(chunk_number):
    """Process a chunk concurrently, saving each result immediately."""

    if not isinstance(chunk_number, int) or not 1 <= chunk_number <= NUM_CHUNKS:
        raise ValueError("chunk_number must be between 1 and 10.")

    total = len(albums_to_process)

    start = (chunk_number - 1) * total // NUM_CHUNKS
    end = chunk_number * total // NUM_CHUNKS

    chunk = albums_to_process.iloc[start:end].copy()

    filename = os.path.join(
        OUTPUT_DIR,
        f"album_art_chunk_{chunk_number}.csv"
    )

    # Create an empty CSV if one doesn't exist
    if not os.path.exists(filename):
        chunk.iloc[0:0].to_csv(filename, index=False)

    # Identify previously completed albums
    if os.path.getsize(filename) > 0:
        existing = pd.read_csv(filename, dtype={"id": str})
        completed_ids = set(existing["id"].dropna())
    else:
        completed_ids = set()

    # Exclude already processed albums
    remaining = chunk[
        ~chunk["id"].astype(str).isin(completed_ids)
    ]

    print(f"\nChunk {chunk_number}")
    print(f"Total albums: {len(chunk)}")
    print(f"Already completed: {len(completed_ids)}")
    print(f"Remaining: {len(remaining)}")

    if remaining.empty:
        print("Nothing left to process!")
        return pd.read_csv(filename)

    # Process concurrently, but save results in the main thread
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = {
            executor.submit(process_one_album, row): row.id
            for row in remaining.itertuples(index=False)
        }

        with tqdm(total=len(futures), desc=f"Chunk {chunk_number}") as pbar:

            for future in as_completed(futures):
                result = future.result()

                # Save immediately after each album finishes
                pd.DataFrame([result]).to_csv(
                    filename,
                    mode="a",
                    header=False,
                    index=False
                )

                pbar.update(1)

    print(f"Chunk {chunk_number} completed!")

    return pd.read_csv(filename)


def combine_album_chunks():
    """Combine all completed chunk CSVs."""

    filenames = [
        os.path.join(
            OUTPUT_DIR,
            f"album_art_chunk_{i}.csv"
        )
        for i in range(1, NUM_CHUNKS + 1)
    ]

    missing = [
        filename for filename in filenames
        if not os.path.exists(filename)
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing chunk files: {missing}"
        )

    combined_df = pd.concat(
        [pd.read_csv(f) for f in filenames],
        ignore_index=True
    )

    combined_df.to_csv(
        "album_art_lookup_results.csv",
        index=False
    )

    print(f"Combined {len(combined_df)} results.")

    return combined_df

Processing Lahai by Sampha


c:\Users\alexw\miniforge3\envs\DS_MAIN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MusicBrainz HTTP 503. Retrying after 10 seconds.
MusicBrainz HTTP 503. Retrying after 20 seconds.
Processing melodic nostalgia by tomcbumpz
Processing OK ORCHESTRA by AJR
MusicBrainz HTTP 503. Retrying after 10 seconds.
Processing HELLMODE by Jeff Rosentock
MusicBrainz HTTP 503. Retrying after 10 seconds.
MusicBrainz HTTP 503. Retrying after 20 seconds.
Processing Dorothy's Harp by Dorothy Ashby
Processing Archive 20172019 by Ugly(UK)
MusicBrainz HTTP 503. Retrying after 10 seconds.
Processing Paloma 1 by Ljubomir Milic
Processing Come to my Garden by Minnie Riperton
Processing Heaven Is A Junkyard by Youth Lagoon
Processing I killed Your Dog by L'rain
Processing Open Season by The High Highs.
Processing Foulden Road by The Neue Grafik Ensemble
Processing Flamboyant by Dorian Electra
Processing Save Rock and Roll by Fallout Boy
MusicBrainz HTTP 503. Retrying after 10 seconds.
Processing The Sale by Julian Chang
Processing Jules by Julian Chang
Processing Dawn FM by The Weeknd
Processin

KeyboardInterrupt: 

In [ ]:
#Handoff:
# Project Handoff: PostgreSQL Album Cover Art Lookup Pipeline
# 1. Project goal

# I have a PostgreSQL database containing album reviews and metadata. I want to automatically find album cover art URLs for every album in the database.

# The database table is merged23_25, with these relevant columns:

# id: internal PostgreSQL primary key
# album_name: album title
# artist_name: artist name

# The database is named album_reviews.

# I am developing and testing the pipeline in Python, likely in a Jupyter notebook. The goal is to identify the correct album release, verify that front-cover artwork is available, and return a usable artwork URL for each album.

# Important constraint: The current script must not modify PostgreSQL. I want to get the lookup process working reliably before implementing any database writes.

# 2. Current technology stack
# Python
# PostgreSQL
# SQLAlchemy
# pandas
# requests
# RapidFuzz
# MusicBrainz release-search API
# Cover Art Archive API

# The main API endpoints are:

# MusicBrainz release search: https://musicbrainz.org/ws/2/release/
# Cover Art Archive release artwork: https://coverartarchive.org/release/{release_id}/front-500

# The artwork URL uses the MusicBrainz release ID.

# 3. Current pipeline design

# The script currently does the following:

# Connects to PostgreSQL using SQLAlchemy.
# Reads a small test sample of three albums from merged23_25, ordered